# N134 · 坐标压缩与秩

**目标：** 保持相对次序而不误保留原始距离。

**先修：** N037, N042。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 去重排序；值到秩；离线预处理；相等键；下界查询；秩与差值区别。

## 从问题到算法

坐标压缩保留的是相等关系与大小顺序，不保留数值差。把巨大、稀疏或负坐标映射为连续秩，可以用数组数据结构处理。右侧更小数先用排序列表和二分写清楚查询含义：lower_bound位置是严格小于当前值的个数，后续再用Fenwick消除列表插入的线性代价。

## 最小实现

**本章接口：** `coordinate_compress(values)`、`rank_transform(arr)`、`count_smaller_sorted_list(nums)`

In [1]:
from bisect import bisect_left,insort

def coordinate_compress(values):
    ordered=sorted(set(values)); rank={v:i for i,v in enumerate(ordered)}
    return ordered,[rank[v] for v in values]

def rank_transform(arr):
    ordered,ranks=coordinate_compress(arr); return [r+1 for r in ranks]

def count_smaller_sorted_list(nums):
    ordered=[]; answer=[]
    for x in reversed(nums): answer.append(bisect_left(ordered,x)); insort(ordered,x)
    return answer[::-1]

## 正确性、前提与复杂度

排序去重产生严格递增的一一映射，故比较关系保持不变；重复值共享同一秩。右向左处理时排序列表恰包含当前位置右侧所有值，二分左界排除了等值元素。

**代价：** 压缩O(n log n)时间O(n)空间；列表计数基线最坏O(n²)时间，因为insort插入要移动元素，二分O(log n)不代表整个操作O(log n)。返回(排序原值表,与输入对应的零基秩)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[10**12,-7,20,-7]; ordered,ranks=coordinate_compress(a)
show_table(['原值','零基秩'],list(zip(a,ranks)))
print('秩不能保持差值：',ordered[-1]-ordered[0],'≠',len(ordered)-1)

原值,零基秩
1000000000000,2
-7,0
20,1
-7,0


秩不能保持差值： 1000000000007 ≠ 2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert rank_transform([40,10,20,30])==[4,1,2,3]
assert rank_transform([7,7,-2])==[2,2,1]
assert count_smaller_sorted_list([5,2,6,1])==[2,1,1,0]
from itertools import product
for n in range(7):
    for a in product((-100,0,10**12),repeat=n):
        ordered,ranks=coordinate_compress(a)
        assert [ordered[r] for r in ranks]==list(a)
        assert all((a[i]<a[j])==(ranks[i]<ranks[j]) for i in range(n) for j in range(n))
        assert count_smaller_sorted_list(a)==[sum(y<x for y in a[i+1:]) for i,x in enumerate(a)]
print('N134: 所有本章断言通过')

N134: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造压缩后距离改变的例子。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 明确有序列表插入仍然线性。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1331. Rank Transform of an Array（canonical）
- 315. Count of Smaller Numbers After Self（preview）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。